# Lab 8: Einführung in Deep Learning mit PyTorch

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Datenordner finden: Notebook liegt in labs/ oder loesungen/, die Daten in data/
DATA = next(p for p in [Path("data"), Path("../data"), Path("../../data")] if p.exists())
print("Datenordner:", DATA)

Dieses Lab gehört zu **Teil 8: Einführung in Deep Learning**.

## Lernziele

- Sie legen Tensoren an, rechnen mit ihnen, ändern ihre Form und tauschen Daten mit NumPy aus.
- Sie berechnen Gradienten mit Autograd und prüfen das Ergebnis von Hand.
- Sie laden FashionMNIST, sehen sich Bilder an und lesen die Form eines Batches.
- Sie schreiben ein Netz als `nn.Module`, vervollständigen die Trainingsschleife und trainieren drei Epochen.
- Sie messen die Genauigkeit auf den Testdaten, finden die verwechselten Klassen und speichern das Modell mit `state_dict`.

Alles läuft auf der CPU. Das Training über drei Epochen dauert je nach Rechner zwischen wenigen Sekunden und etwa einer Minute. Die Kontrollergebnisse gelten für `torch.manual_seed(42)` an den angegebenen Stellen. Kleine Abweichungen in der letzten Stelle sind je nach Rechner normal.

In [ ]:
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

print("torch", torch.__version__)

## Block 1: Tensoren

Ein Tensor verhält sich wie ein NumPy-Array. Zusätzlich kann er Gradienten mitführen.

1. Legen Sie die Matrizen `a = [[1, 2], [3, 4]]` und `b = [[5, 6], [7, 8]]` als Tensoren mit Kommazahlen an. Berechnen Sie Summe, elementweises Produkt und Matrizenprodukt und prüfen Sie Form und Datentyp. **Erwartet:** `a + b` ergibt [[6, 8], [10, 12]], `a * b` ergibt [[5, 12], [21, 32]], `a @ b` ergibt [[19, 22], [43, 50]], Form `torch.Size([2, 2])`, Datentyp `torch.float32`.
2. Ändern Sie die Form: Legen Sie mit `torch.arange(12)` die Zahlen 0 bis 11 an, formen Sie sie zu 3 Zeilen und 4 Spalten um und danach zu einem Stapel aus 2 „Bildern" mit 2 x 3 Pixeln. Machen Sie aus dem Stapel mit `nn.Flatten()` wieder eine Zeile je Bild. **Erwartet:** Formen `[3, 4]`, `[2, 2, 3]` und `[2, 6]`.
3. Wandeln Sie das NumPy-Array `arr` in einen Tensor um, prüfen Sie den Datentyp, stellen Sie ihn auf `float32` um und wandeln Sie das Ergebnis zurück in ein NumPy-Array. **Erwartet:** erst `torch.float64`, nach `.float()` `torch.float32`, am Ende ein `numpy.ndarray` mit dtype `float32`.

In [ ]:
# Aufgabe 1: Tensoren anlegen und rechnen
# Tipp: torch.tensor([[1., 2], [3, 4]]), der Punkt hinter der 1 macht daraus Kommazahlen
a = ...
b = ...

# print(a + b)
# print(a * b)      # elementweise
# print(a @ b)      # Matrizenmultiplikation
# print(a.shape, a.dtype)

In [ ]:
# Aufgabe 2: Form ändern
# Tipp: reshape(zeilen, spalten); nn.Flatten() lässt die erste Dimension (den Batch) stehen
t = torch.arange(12)
m = ...          # 3 Zeilen, 4 Spalten
stapel = ...     # 2 Bilder mit 2 x 3 Pixeln
flach = ...      # je Bild eine Zeile

# print(m.shape, stapel.shape, flach.shape)

In [ ]:
# Aufgabe 3: NumPy -> Tensor -> NumPy
# Tipp: torch.from_numpy, .float(), .numpy()
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
t64 = ...
t32 = ...
zurueck = ...

# print(t64.dtype, t32.dtype)
# print(type(zurueck), zurueck.dtype)

## Block 2: Autograd

Mit `requires_grad=True` merkt sich PyTorch jede Rechnung. `backward()` läuft die Rechnungen rückwärts ab und legt die Ableitung in `.grad`.

1. Nachmachen: Berechnen Sie für `y = x**2 + 3*x + 5` den Gradienten bei x = 2.0. **Erwartet:** `tensor(7.)`. Von Hand: f'(x) = 2x + 3, also 2 * 2 + 3 = 7.
2. Abwandeln: Berechnen Sie für `y = 3*x**2 - 4*x` den Gradienten bei x = 1.0 und schreiben Sie die Handrechnung als Kommentar dazu. **Erwartet:** `tensor(2.)`.
3. Selbst lösen: Ein Neuron mit den Eingaben x = (1, 2, 3), den Gewichten w = (0.5, -1, 0.2) und dem Bias b = 0.1 soll den Zielwert 1 treffen. Berechnen Sie `z = w @ x + b`, den Verlust `(z - 1)**2` und die Gradienten nach `w` und `b`. Prüfen Sie von Hand mit der Formel 2 * (z - 1) * x. **Erwartet:** z = -0.8, Verlust 3.24, Gradient nach w = (-3.6, -7.2, -10.8), Gradient nach b = -3.6.

In [ ]:
# Aufgabe 1: Gradient von y = x**2 + 3*x + 5 bei x = 2.0
# Tipp: requires_grad=True, dann y.backward() und x.grad
x = torch.tensor(2.0, requires_grad=True)
# y = ...
# y.backward()
print(x.grad)

In [ ]:
# Aufgabe 2: Gradient von y = 3*x**2 - 4*x bei x = 1.0
# Handrechnung: f'(x) = ...
x = ...
# y = ...
# y.backward()
# print(x.grad)

In [ ]:
# Aufgabe 3: Gradienten eines Neurons nach Gewichten und Bias
# Tipp: Nur w und b brauchen requires_grad=True. Die Eingaben kommen aus den Daten.
x = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([0.5, -1.0, 0.2], requires_grad=True)
b = torch.tensor(0.1, requires_grad=True)

# z = ...
# verlust = ...
# verlust.backward()
# print("z:", z.item(), "Verlust:", verlust.item())
print("Gradient w:", w.grad)
print("Gradient b:", b.grad)
# print("von Hand:  ", 2 * (z.item() - 1) * x)

## Block 3: FashionMNIST laden und ansehen

Die komprimierten Dateien liegen im Datenordner des Kurses. `download=True` lädt deshalb nichts aus dem Netz, sondern entpackt nur. Wichtig ist `root=str(DATA)`: So findet das Notebook die Dateien, egal aus welchem Ordner es gestartet wurde.

1. Laden Sie Trainings- und Testdaten mit `transform=ToTensor()`. **Erwartet:** 60000 Trainingsbilder und 10000 Testbilder.
2. Holen Sie das erste Trainingsbild mit seinem Label, geben Sie Form, kleinsten und größten Pixelwert aus und zeigen Sie das Bild mit dem deutschen Klassennamen als Titel. **Erwartet:** Form `[1, 28, 28]`, Werte von 0.0 bis 1.0, Label 9, Klasse „Stiefelette".
3. Legen Sie `train_loader` (mit Mischen) und `test_loader` (ohne Mischen) mit `batch_size=64` an. Geben Sie die Zahl der Batches und die Form eines Batches aus. **Erwartet:** 938 und 157 Batches, Form der Bilder `[64, 1, 28, 28]`, Form der Labels `[64]`.

In [ ]:
# Aufgabe 1: Daten laden
# Tipp: Die Trainingsdaten stehen schon da. Für die Testdaten ändert sich nur train=False.
train_data = datasets.FashionMNIST(root=str(DATA), train=True, download=True, transform=ToTensor())
test_data = ...

# print(len(train_data), len(test_data))

In [ ]:
# Die zehn Klassen auf Deutsch, in der Reihenfolge der Labels 0 bis 9
klassen = ["T-Shirt/Top", "Hose", "Pullover", "Kleid", "Mantel",
           "Sandale", "Hemd", "Sneaker", "Tasche", "Stiefelette"]

In [ ]:
# Aufgabe 2: erstes Bild ansehen
# Tipp: train_data[0] liefert ein Paar (bild, label); imshow braucht 28 x 28, also bild[0]
# bild, label = ...
# print(bild.shape, bild.min().item(), bild.max().item(), label)

fig, ax = plt.subplots(figsize=(3, 3))
# ax.imshow(..., cmap="gray")
# ax.set_title(...)
ax.axis("off")
plt.show()

In [ ]:
# Aufgabe 3: DataLoader anlegen, Form eines Batches ansehen
# Tipp: Der test_loader sieht genauso aus, nur mit test_data und shuffle=False
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)    # mischt in jeder Epoche neu
test_loader = ...

# print(len(train_loader), len(test_loader))
bilder, labels = next(iter(train_loader))
# print(bilder.shape, labels.shape)

In [ ]:
# Zur Orientierung: von jeder der zehn Klassen das erste Trainingsbild
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for nummer, ax in enumerate(axes.flat):
    index = int((train_data.targets == nummer).nonzero()[0])
    ax.imshow(train_data[index][0][0], cmap="gray")
    ax.set_title(f"{nummer}: {klassen[nummer]}")
    ax.axis("off")
plt.show()

## Block 4: Das Netz als `nn.Module`

Das Netz soll so aussehen: 784 Pixelwerte, eine versteckte Schicht mit 128 Neuronen und ReLU, 10 Ausgänge (Logits, ohne Softmax).

1. In der Zelle steht ein lauffähiges Netz mit nur **einer** Linearschicht (784 auf 10). Bauen Sie es um: `layer1` führt von 784 auf 128, eine neue `layer2` von 128 auf 10, im `forward` steht zwischen beiden `torch.relu`. Zählen Sie danach die Parameter. **Erwartet:** 101770 (vor dem Umbau: 7850).
2. Rechnen Sie die Zahl der Parameter von Hand nach und geben Sie die Zahl je Schicht aus. **Erwartet:** `layer1.weight` 100352, `layer1.bias` 128, `layer2.weight` 1280, `layer2.bias` 10.
3. Schicken Sie den Batch `bilder` aus Block 3 durch das untrainierte Netz. Welche Form hat die Ausgabe? Welche Klassen sagt das Netz für die ersten acht Bilder vorher? **Erwartet:** Form `[64, 10]`. Die Vorhersagen sind noch geraten.

In [ ]:
# Aufgabe 1: Netz auf zwei Linearschichten umbauen (784 -> 128 -> 10)
# Tipp: In __init__ stehen die Schichten, in forward der Weg der Daten durch das Netz
class Netz(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()              # 1x28x28 -> 784
        self.layer1 = nn.Linear(28 * 28, 10)     # ändern: 784 -> 128
        # ergänzen: self.layer2 mit 128 -> 10

    def forward(self, x):
        x = self.flatten(x)
        return self.layer1(x)                    # ändern: ReLU auf layer1, danach layer2

torch.manual_seed(42)
model = Netz()
print(sum(p.numel() for p in model.parameters()))

In [ ]:
# Aufgabe 2: Parameter je Schicht, dazu die Handrechnung
# Tipp: model.named_parameters() liefert Paare (name, tensor), tensor.numel() zählt die Werte
for name, p in model.named_parameters():
    pass   # print(name, ..., ...)

von_hand = ...   # Formel mit 784, 128 und 10
von_hand

In [ ]:
# Aufgabe 3: ein Batch durch das untrainierte Netz
# Tipp: model(bilder), nie model.forward(bilder); argmax(dim=1) wählt je Bild den größten Logit
outputs = ...

# print(outputs.shape)
# print("vorhergesagt:", outputs.argmax(dim=1)[:8].tolist())
# print("wahr:        ", labels[:8].tolist())

## Block 5: Trainingsschleife

Die Schleife hat vier Schritte: Vorwärtslauf, Verlust, Rückwärtslauf, Schritt des Optimierers. Davor werden die alten Gradienten gelöscht.

1. In der Funktion `trainiere` fehlen drei Zeilen. Ergänzen Sie sie an den markierten Stellen und trainieren Sie drei Epochen mit Adam und `lr=1e-3`. **Erwartet:** Verlust je Epoche 0.546, 0.402, 0.358. Bleibt Ihr Verlust bei etwa 2.3 stehen, fehlen die drei Zeilen noch: Das Netz rät dann weiter.
2. Zeichnen Sie den Verlust je Epoche und rechnen Sie aus, wie viele Schritte der Optimierer insgesamt gemacht hat. **Erwartet:** fallende Linie, 3 * 938 = 2814 Schritte.
3. Lernrate ausprobieren: Trainieren Sie je ein frisches Netz **eine** Epoche lang mit `lr=1e-1` und mit `lr=1e-5` und vergleichen Sie mit dem Wert 0.546 aus Aufgabe 1. **Erwartet:** Beide Läufe enden weit über 0.546. Mit 1e-1 bleibt der Verlust hoch (um 1.9), weil die Schritte zu groß sind. Mit 1e-5 sinkt er nur langsam (um 1.8 nach einer Epoche), weil die Schritte zu klein sind.

In [ ]:
# Aufgabe 1: drei Zeilen ergänzen, dann drei Epochen trainieren
def trainiere(model, optimizer, n_epochs=3):
    loss_fn = nn.CrossEntropyLoss()              # mehrere Klassen
    verluste = []
    for epoch in range(n_epochs):
        model.train()                            # Trainingsmodus
        running_loss = 0.0
        for bilder, labels in train_loader:
            # ERGÄNZEN 1: alte Gradienten löschen
            outputs = model(bilder)              # Vorwärtslauf
            loss = loss_fn(outputs, labels)      # Verlust
            # ERGÄNZEN 2: Rückwärtslauf
            # ERGÄNZEN 3: Gewichte anpassen
            running_loss += loss.item()
        verluste.append(running_loss / len(train_loader))
        print(f"Epoche {epoch + 1}/{n_epochs}, Verlust: {verluste[-1]:.4f}")
    return verluste

torch.manual_seed(42)
model = Netz()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
verluste = trainiere(model, optimizer, n_epochs=3)

In [ ]:
# Aufgabe 2: Verlust je Epoche zeichnen, Zahl der Schritte ausrechnen
# Tipp: ein Schritt je Batch, len(train_loader) Batches je Epoche
schritte = ...
print("Schritte des Optimierers:", schritte)

fig, ax = plt.subplots()
# ax.plot(...)
ax.set_xlabel("Epoche")
ax.set_ylabel("mittlerer Verlust")
plt.show()

In [ ]:
# Aufgabe 3: Lernrate 1e-1 und 1e-5, je ein frisches Netz, je eine Epoche
# Tipp: vor jedem Netz torch.manual_seed(42); das trainierte model aus Aufgabe 1 nicht überschreiben
for lr in [1e-1, 1e-5]:
    print("Lernrate", lr)
    # torch.manual_seed(42)
    # netz_lr = ...
    # trainiere(netz_lr, ..., n_epochs=1)

## Block 6: Auswerten, Verwechslungen ansehen, speichern

Die nächste Zelle enthält die Auswertungsschleife von den Folien, als Funktion verpackt. Sie liefert für einen ganzen `DataLoader` die vorhergesagten und die wahren Klassen.

1. Berechnen Sie die Genauigkeit des trainierten `model` auf den Testdaten. **Erwartet:** 86.33 %. Ein untrainiertes Netz läge bei etwa 10 %.
2. Bilden Sie die Verwechslungstabelle (Zeilen: wahre Klasse, Spalten: vorhergesagte Klasse) und die Genauigkeit je Klasse. Welche Klasse ist am schwierigsten, und womit wird sie verwechselt? **Erwartet:** Am schwierigsten ist „Hemd" mit 0.66, dicht gefolgt von „Pullover" mit 0.68. Hemden werden vor allem für T-Shirt/Top (141 von 1000), Mantel (67) und Pullover (61) gehalten, Pullover vor allem für Mantel (155). Am besten erkannt werden Tasche, Hose und Stiefelette (0.96 bis 0.98). Verwechselt werden also Oberteile untereinander und Schuhe untereinander, nie Hosen mit Taschen.
3. Speichern Sie die Gewichte mit `state_dict` in ein temporäres Verzeichnis, laden Sie sie in ein neues Objekt `model2` und prüfen Sie, dass die Genauigkeit gleich bleibt. **Erwartet:** vier Einträge im `state_dict`, Genauigkeit von `model2` wieder 86.33 %.

In [ ]:
def vorhersagen(model, loader):
    """Liefert zwei Tensoren: vorhergesagte und wahre Klassen für alle Bilder des Loaders."""
    model.eval()                                 # Auswertungsmodus
    alle_pred, alle_wahr = [], []
    with torch.no_grad():                        # keine Gradienten nötig
        for bilder, labels in loader:
            outputs = model(bilder)
            alle_pred.append(outputs.argmax(dim=1))   # Klasse mit dem höchsten Logit
            alle_wahr.append(labels)
    return torch.cat(alle_pred), torch.cat(alle_wahr)

In [ ]:
# Aufgabe 1: Genauigkeit auf den Testdaten
# Tipp: (pred == wahr) ist ein Tensor aus True/False; .sum().item() zählt die Treffer
# pred, wahr = vorhersagen(model, test_loader)
correct = ...
total = ...

# print(f"Genauigkeit: {correct / total * 100:.2f} %")

In [ ]:
# Aufgabe 2: Verwechslungstabelle und Genauigkeit je Klasse
# Tipp: pd.crosstab(wahre Klassen, vorhergesagte Klassen); die Treffer stehen auf der Diagonalen
# wahr_name = pd.Series(wahr.numpy(), name="wahr").map(lambda i: klassen[i])
# pred_name = ...
tabelle = ...
tabelle

# je_klasse = ...   # Diagonale geteilt durch Zeilensumme
# print(je_klasse.sort_values().round(2))

In [ ]:
# Aufgabe 3: speichern, in ein neues Objekt laden, Genauigkeit vergleichen
# Tipp: torch.save(model.state_dict(), pfad), model2.load_state_dict(torch.load(pfad))
print(list(model.state_dict().keys()))

with tempfile.TemporaryDirectory() as ordner:
    pfad = Path(ordner) / "model.pth"
    # torch.save(...)
    # model2 = Netz()
    # model2.load_state_dict(...)

# pred2, wahr2 = vorhersagen(model2, test_loader)
# print(f"Genauigkeit model2: {(pred2 == wahr2).sum().item() / len(wahr2) * 100:.2f} %")

## Zusatzaufgaben

Für die ersten beiden Aufgaben steht unten die Klasse `NetzFlex`: dasselbe Netz, aber mit einstellbarer Größe der versteckten Schicht und optionalem Dropout.

1. **Breitere Schicht.** Trainieren Sie `NetzFlex(hidden=256)` drei Epochen (Seed 42, Adam, `lr=1e-3`) und vergleichen Sie Parameterzahl und Testgenauigkeit mit dem Netz aus Block 5. **Erwartet:** 203530 Parameter, Verlust 0.516, 0.380, 0.342, Genauigkeit 86.24 % (vorher 86.33 %). Der Verlust auf den Trainingsdaten ist etwas niedriger, die Genauigkeit auf den Testdaten praktisch gleich: Doppelt so viele Parameter bringen nach drei Epochen nichts.
2. **Dropout.** Trainieren Sie `NetzFlex(hidden=128, dropout=0.3)` drei Epochen und vergleichen Sie. **Erwartet:** Verlust 0.594, 0.429, 0.392, Genauigkeit 86.16 %. Der Verlust auf den Trainingsdaten ist höher, weil beim Training Neuronen abgeschaltet sind. Nach nur drei Epochen gibt es noch kein Overfitting, deshalb hilft Dropout hier nicht. Seine Wirkung zeigt sich erst bei langem Training.
3. **SGD statt Adam.** Trainieren Sie ein frisches `Netz()` drei Epochen mit `optim.SGD(..., lr=1e-2)`. **Erwartet:** Verlust 1.130, 0.675, 0.579, Genauigkeit 80.09 %. SGD mit fester Lernrate kommt in drei Epochen deutlich langsamer voran als Adam.
4. **Nur mit Netz: Hugging Face `datasets`.** Dieselben Bilder liegen auch im Hugging Face Hub. Die Zelle lädt sie nur, wenn Sie `MIT_NETZ = True` setzen und Ihr Rechner huggingface.co erreicht. Ohne Netz überspringen Sie die Aufgabe. **Erwartet (mit Netz):** 60000 Trainings- und 10000 Testbilder, erstes Bild mit Form `[1, 28, 28]`, Datentyp `torch.uint8` und Label 9, Pixelwerte als Ganzzahlen von 0 bis 255. Je nach Version von `datasets` kann die Form auch `[28, 28]` sein.

In [ ]:
class NetzFlex(nn.Module):
    def __init__(self, hidden=128, dropout=0.0):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(28 * 28, hidden)
        self.drop = nn.Dropout(dropout)          # schaltet beim Training zufällig Neuronen ab
        self.layer2 = nn.Linear(hidden, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.layer1(x))
        x = self.drop(x)                         # bei model.eval() ist Dropout aus
        return self.layer2(x)


def genauigkeit(model, loader):
    pred, wahr = vorhersagen(model, loader)
    return (pred == wahr).sum().item() / len(wahr) * 100

In [ ]:
# Zusatz 1: versteckte Schicht mit 256 Neuronen
# Tipp: gleicher Ablauf wie in Block 5: Seed, Netz, Optimierer, trainiere, genauigkeit
torch.manual_seed(42)
netz_256 = ...
# print(sum(p.numel() for p in netz_256.parameters()))
# trainiere(netz_256, ..., n_epochs=3)
# print(f"Genauigkeit: {genauigkeit(netz_256, test_loader):.2f} %")

In [ ]:
# Zusatz 2: Dropout 0.3 nach der versteckten Schicht
torch.manual_seed(42)
netz_drop = ...
# trainiere(netz_drop, ..., n_epochs=3)
# print(f"Genauigkeit: {genauigkeit(netz_drop, test_loader):.2f} %")

In [ ]:
# Zusatz 3: SGD statt Adam
torch.manual_seed(42)
netz_sgd = ...
# trainiere(netz_sgd, optim.SGD(...), n_epochs=3)
# print(f"Genauigkeit: {genauigkeit(netz_sgd, test_loader):.2f} %")

In [ ]:
# Zusatz 4 (nur mit Netz): FashionMNIST aus dem Hugging Face Hub
# Tipp: load_dataset("zalando-datasets/fashion_mnist"), danach ds["train"].with_format("torch")
MIT_NETZ = False     # auf True setzen, wenn huggingface.co erreichbar ist

if MIT_NETZ:
    try:
        from datasets import load_dataset
        # ds = ...
        # print(ds)
    except Exception as fehler:
        print("Kein Zugriff auf den Hub:", type(fehler).__name__)
else:
    print("Übersprungen: MIT_NETZ ist False.")

## Was Sie mitnehmen

- Ein Tensor ist ein Array, das Gradienten mitführen kann. `backward()` berechnet die Ableitungen, die Sie sonst von Hand ausrechnen müssten.
- Die Trainingsschleife besteht aus `optimizer.zero_grad()`, `model(bilder)`, `loss_fn(...)`, `loss.backward()` und `optimizer.step()`. Fehlt einer der drei Aufrufe rund um den Verlust, lernt das Netz nichts oder falsch.
- Ausgewertet wird mit `model.eval()` und `torch.no_grad()` auf Testdaten. Die Genauigkeit je Klasse zeigt mehr als die Gesamtzahl. Gespeichert werden nur die Gewichte (`state_dict`), die Klasse des Netzes muss beim Laden als Code vorliegen.